<a href="https://colab.research.google.com/github/Abdulkadir03/Crop-Recommender-for-Sahel-Region/blob/main/notebooks/05b_ground_truth_crop_harvest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

05b: Ground-truth crop harvest

**Purpose:** Collect independent field-level crop observations to check the model against real fields. These points are used for validation only; training labels come from SPAM (notebook 05).

**Status:** Complete. Household-survey ground truth for Nigeria and Niger is in notebook 05c.

**Inputs (Part A):**
- CropHarvest `labels.geojson` (Zenodo record 10251170, version v14, published 4 Dec 2023; Tseng et al., NeurIPS 2021; CC BY-SA 4.0), downloaded 23 Oct 2026.: 113,893 points globally. Only the label file is used, not the satellite features.
- `grid.parquet` (notebook 01) and `grid_pixel_map.parquet` (notebook 05)

**Method (Part A):**
1. Points inside the study region (-19 to 26 E, 3 to 19 N) are matched to the nearest grid cell and kept if within 2 km (8,295 of 11,674).
2. Crop labels are mapped to SPAM crop codes (e.g. maize to maiz, millet to pmil, groundnut to grou). Unmapped labels (no label, fallow, vegetables, trees) are excluded, and duplicates (same location, year and label) are removed.
3. Points are kept only if their grid cell has a SPAM label pixel.

**Outputs** (Google Drive, not in this repo):
- `ground_truth_cropharvest_clean.parquet`: 5,972 usable points with crop code, grid cell, SPAM pixel, country, dataset and year
- `ground_truth_cropharvest_pixels.parquet`: observed crops per SPAM pixel (93 pixels)

**Notes:**
- The 5,972 points fall in only 472 grid cells and 93 SPAM pixels. Validation statistics must be computed per pixel (e.g. bootstrap over pixels), not per point.
- Coverage is limited to Burkina Faso (44%), Senegal (29%) and Mali (27%); there are no usable points for Nigeria, Niger, Chad, Cameroon, Mauritania or Gambia.
- Pearl millet (72 pixels), maize (66), sorghum (63), groundnut (46) and rice (35) are the best-supported crops; cowpea (19), cotton (14), sesame (12) and soybean (9) are indicative only; the rest are too sparse to evaluate.
- "Millet" is mapped to pearl millet. Points were collected in 2017-2022, while SPAM represents 2019-2021.
- Validation must use out-of-fold predictions from spatial cross-validation; scoring points with a model trained on their own pixel would be leakage.
- The contributing projects (JECAM, Mali) have their own terms; see the data sources table in the README.

In [ ]:
# @title Cell 1: setup
import os, subprocess
import numpy as np, pandas as pd
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive'
RAW = f'{BASE}/crop_project/raw/cropharvest'
os.makedirs(RAW, exist_ok=True)

grid = pd.read_parquet(f'{BASE}/crop_project/grid.parquet')
BBOX = (-19.0, 3.0, 26.0, 19.0)      # lon_min, lat_min, lon_max, lat_max
print(grid.shape)

In [ ]:
# @title Cell 2: get the label file
URL = 'https://zenodo.org/records/10251170/files/labels.geojson?download=1'                                   # paste the link address of labels.geojson
out = f'{RAW}/labels.geojson'
if URL and not os.path.exists(out):
    subprocess.run(['wget', '-q', '-O', out, URL], check=False)
print('labels.geojson:', f'{os.path.getsize(out) / 1e6:.0f} MB' if os.path.exists(out) else 'MISSING')

In [ ]:
# @title Cell 3: what does CropHarvest contain in the region?
import geopandas as gpd

lab = gpd.read_file(f'{RAW}/labels.geojson')
print('All points:', lab.shape, '| columns:', list(lab.columns))

lon0, lat0, lon1, lat1 = BBOX
reg = lab[(lab.lon >= lon0) & (lab.lon <= lon1) & (lab.lat >= lat0) & (lab.lat <= lat1)].copy()
print('\nPoints inside the region box:', len(reg), '| crop points:', int(reg.is_crop.sum()))

print('\nBy source dataset:'); print(reg.groupby('dataset').size().sort_values(ascending=False))
print('\nTop labels:'); print(reg.label.value_counts(dropna=False).head(30))
print('\nCollection year:'); print(reg.collection_date.astype(str).str[:4].value_counts().sort_index())

In [ ]:
# @title Cell 4: match the points to your grid and crop codes
#@title Match CropHarvest points to the grid and to our crop codes { display-mode: "both" }
from sklearn.neighbors import BallTree

MAX_KM = 2.0                     # a point must be within this distance of a grid cell centre

gm = pd.read_parquet(f'{BASE}/crop_project/grid_pixel_map.parquet')      # grid_id, px, py, has_label
g = grid[['grid_id', 'lon', 'lat', 'adm0']].merge(gm, on='grid_id')
tree = BallTree(np.radians(g[['lat', 'lon']].to_numpy(dtype='float64')), metric='haversine')

pts = reg[['index', 'dataset', 'label', 'classification_label', 'is_crop',
           'lat', 'lon', 'collection_date']].copy().reset_index(drop=True)
dist, ix = tree.query(np.radians(pts[['lat', 'lon']].to_numpy(dtype='float64')), k=1)
pts['dist_km'] = dist[:, 0] * 6371.0
near = g.iloc[ix[:, 0]].reset_index(drop=True)
pts = pd.concat([pts, near[['grid_id', 'adm0', 'px', 'py', 'has_label']]], axis=1)
pts = pts[pts.dist_km <= MAX_KM].copy()
pts['year'] = pd.to_datetime(pts.collection_date, errors='coerce').dt.year

CODE = {'maize': 'maiz', 'millet': 'pmil', 'pearl millet': 'pmil', 'sorghum': 'sorg',
        'groundnut': 'grou', 'groundnuts': 'grou', 'cotton': 'cott', 'rice': 'rice',
        'soybean': 'soyb', 'cowpea': 'cowp', 'sesame': 'sesa', 'tomato': 'toma',
        'cassava': 'cass', 'bean': 'bean', 'onion': 'onio', 'yam': 'yams',
        'sweet potato': 'swpo', 'sugarcane': 'sugc'}
pts['crop_code'] = pts.label.astype('string').str.lower().map(CODE)
pts = pts.drop_duplicates(['lat', 'lon', 'year', 'label']).reset_index(drop=True)

print(len(pts), 'points within', MAX_KM, 'km of a grid cell (of', len(reg), 'in the box)')
print('\nBy country:'); print(pts.groupby('adm0').size().sort_values(ascending=False))
print('\nBy source dataset:'); print(pts.groupby('dataset').size().sort_values(ascending=False))
print('\nPoints per mapped crop:'); print(pts.crop_code.value_counts())
print('\nLargest labels not mapped to a crop code:')
print(pts[pts.crop_code.isna()].label.value_counts(dropna=False).head(15))
print('\nPoints whose grid cell has a SPAM label pixel:', int(pts.has_label.sum()))

pts.to_parquet(f'{BASE}/crop_project/ground_truth_cropharvest.parquet', index=False)
from google.colab import drive
drive.flush_and_unmount(); drive.mount('/content/drive')

In [ ]:
# @title Cell 5: effective validation sample
# It needs pts from Cell 4. If Colab disconnected, run Cells 1 and 3 and then Cell 4 first.
ok = pts[pts.crop_code.notna() & pts.has_label].copy()
ok['pixel'] = ok.px.astype(str) + '_' + ok.py.astype(str)

eff = (ok.groupby('crop_code')
         .agg(points=('index', 'size'), grid_cells=('grid_id', 'nunique'),
              spam_pixels=('pixel', 'nunique'), countries=('adm0', 'nunique'),
              datasets=('dataset', 'nunique'))
         .sort_values('points', ascending=False))
print(eff)
print('\nUsable points:', len(ok), '| distinct grid cells:', ok.grid_id.nunique(),
      '| distinct SPAM pixels:', ok.pixel.nunique())
print('\nShare of usable points by country (%):'); print((ok.adm0.value_counts(normalize=True) * 100).round(1))
print('\nShare of usable points by dataset (%):'); print((ok.dataset.value_counts(normalize=True) * 100).round(1))
print('\nCollection years:'); print(ok.year.value_counts().sort_index())

# Observed crops per SPAM pixel: what model rankings will later be compared against
obs = ok.groupby(['px', 'py', 'crop_code']).size().rename('n_points').reset_index()
print('\nPixels with at least one validation crop:', obs[['px', 'py']].drop_duplicates().shape[0])

ok.drop(columns='pixel').to_parquet(f'{BASE}/crop_project/ground_truth_cropharvest_clean.parquet', index=False)
obs.to_parquet(f'{BASE}/crop_project/ground_truth_cropharvest_pixels.parquet', index=False)
drive.flush_and_unmount(); drive.mount('/content/drive')